# Mixed models. Also known as multilevel models, also known as hierarchical models

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iknyazeva/bayes-cogsci-book/blob/main/notebooks/colab/09_mixed_models_linear.ipynb)
*Russian version: [09_mixed_models_linear_ru.ipynb](https://colab.research.google.com/github/iknyazeva/bayes-cogsci-book/blob/main/notebooks/colab/09_mixed_models_linear_ru.ipynb)*

This workbook accompanies Lesson 9 (Repeated Observations & Hierarchical Models) and the "Classical Mixed Models vs. Hierarchical Bayes" page. Using the `sleepstudy` sleep-deprivation data, we go all the way from ordinary linear regression to random-effects models and then to Bayesian hierarchical models in Bambi.


In [ ]:
import sys, subprocess

try:
    import pingouin as pg
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pingouin"])
    import pingouin as pg

import arviz as az
import seaborn as sns
import bambi as bmb
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
from statsmodels.stats.stattools import durbin_watson
from statsmodels.graphics.gofplots import qqplot

RANDOM_SEED = 2026

az.style.use("arviz-darkgrid")

# Данные: sleepstudy (Belenky et al., 2003) — каноническая копия в репозитории курса
import os, urllib.request
DATA_PATH = "data/sleepstudy.csv"
os.makedirs("data", exist_ok=True)
if not os.path.exists(DATA_PATH):
    url = "https://raw.githubusercontent.com/iknyazeva/bayes-cogsci-book/main/data/sleepstudy.csv"
    urllib.request.urlretrieve(url, DATA_PATH)


When we discussed the conditions that a linear model has to satisfy, we singled out the requirement that observations be independent of each other. This is a genuinely important requirement: when it is violated, the model's estimates are incorrect and no valid conclusions can be drawn.

Very often, however, observations are grouped by some attribute:
- the same measurements taken at different time points;
- measurements collected in several organizations (for example, studies run in several schools);
- repeated measurements for the same person.

That is, any situation where there is reason to expect clusters and within-cluster correlation.

Let us start with an example available in many packages.

## Sleep deprivation and reaction time

Experiment: on the night before the study every subject got a full night of sleep; on each of the following 9 nights only 3 hours. Every day, reaction time was measured in a series of tests.

Research question: how does reaction time change with the duration of sleep deprivation.


In [ ]:
data = pd.read_csv(DATA_PATH)
display(data.head())
data.info()


In [ ]:
data['Subject'] = data.Subject.astype("category")


In [ ]:
data.groupby('Days')[['Reaction']].mean()


In [ ]:
#посмотрим сколько людей и по сколько на каждого приходится значений
plt.figure(figsize=(10,3))
data.Subject.value_counts().plot(kind='bar');


In [ ]:
plt.scatter(data.Days, data.Reaction, color="C0", ec="black", alpha=0.7);


In [ ]:
plt.figure(figsize=(15,8))
ax = sns.scatterplot(x = 'Days', y = 'Reaction', hue = 'Subject', data=data)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))


In [ ]:
def plot_data(data):
    fig, axes = plt.subplots(2, 9, figsize=(16, 7.5), sharey=True, sharex=True, dpi=100, constrained_layout=False)
    fig.subplots_adjust(left=0.075, right=0.975, bottom=0.075, top=0.925, wspace=0.03)

    axes_flat = axes.ravel()

    for i, subject in enumerate(data["Subject"].unique()):
        ax = axes_flat[i]
        idx = data.index[data["Subject"] == subject].tolist()
        days = data.loc[idx, "Days"].values
        reaction = data.loc[idx, "Reaction"].values

        # Plot observed data points
        ax.scatter(days, reaction, color="C0", ec="black", alpha=0.7)

        # Add a title
        ax.set_title(f"Subject: {subject}", fontsize=14)

    ax.xaxis.set_ticks([0, 2, 4, 6, 8])
    fig.text(0.5, 0.02, "Days", fontsize=14)
    fig.text(0.03, 0.5, "Reaction time (ms)", rotation=90, fontsize=14, va="center")

    return axes


In [ ]:
plot_data(data);


Our main goal is to measure the relationship between days of sleep deprivation and reaction time. We care both about the overall effect shared by all subjects and about the effect specific to each individual subject. For such tasks there are special models: hierarchical regression models. Within these models one can account for both the common effects and the subject-specific ones. In the statistical literature such effects are also known as fixed and random effects. But first let us fit an ordinary linear model with tools you already know.


### Ordinary linear model
- Model 1 — first we fit a simple model of reaction time on days of deprivation;
- Model 2 — then a more complex model that adds the subject to the model.

#### Model 1


In [ ]:
lm_basic = smf.ols(formula="Reaction ~ Days", data=data).fit()
print(lm_basic.summary())

print("\nDurbin-Watson:", round(durbin_watson(lm_basic.resid), 3))


Looking at the table, the only alarm bell is Durbin-Watson. An important clarification: the Durbin-Watson statistic tests the **autocorrelation of residuals** in row order, not collinearity of predictors. A value near 2 means no autocorrelation; ours is ≈ 0.7 — strong positive autocorrelation: consecutive rows belong to the same subject, so their errors are similar. A rough rule of thumb: values between about 1 and 3 are usually unremarkable, clear departures from 2 call for investigation. Let us look at the residuals. A Shapiro test would reject normality, but the main problem here is the structure of the residuals, not the shape of their distribution.


In [ ]:
fig, axs = plt.subplots(1, 3, figsize = (10, 3))
axs[0].hist(lm_basic.resid)
qqplot(lm_basic.resid, line="s", ax=axs[1], markersize=3)
axs[2].scatter(range(len(lm_basic.resid)), lm_basic.resid, s=8)
plt.show()
pg.normality(lm_basic.resid)


In [ ]:
pred_ols = lm_basic.get_prediction()
iv_l = pred_ols.summary_frame()["obs_ci_lower"]
iv_u = pred_ols.summary_frame()["obs_ci_upper"]

mean_iv_l = pred_ols.summary_frame()["mean_ci_lower"]
mean_iv_u = pred_ols.summary_frame()["mean_ci_upper"]

order = np.argsort(data.Days.values)
fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(data.Days, data.Reaction, color="C0", ec="black", alpha=0.7, label="data")
ax.plot(data.Days.values[order], lm_basic.fittedvalues.values[order], "r--.", label="OLS")
ax.plot(data.Days.values[order], mean_iv_l.values[order], "r--", label='Mean 95% CI: lower')
ax.plot(data.Days.values[order], mean_iv_u.values[order], "r--", label='Mean 95% CI: upper')
ax.plot(data.Days.values[order], iv_l.values[order], "b--", label='Obs 95%: lower')
ax.plot(data.Days.values[order], iv_u.values[order], "g--", label='Obs 95%: upper')
ax.legend(loc="best");


#### Model 2

In this model we give every subject their own "correction" to the intercept of the regression equation (dummy variables for subjects). The model loses universality: predictions are only possible for a specific, known subject. But the model improves, and the problem of correlated residuals is essentially resolved.


In [ ]:
lm_sub = smf.ols(formula="Reaction ~ Days + C(Subject)", data=data).fit()
print(lm_sub.summary())


In [ ]:
fig, axs = plt.subplots(1, 3, figsize = (10, 3))
axs[0].hist(lm_sub.resid)
qqplot(lm_sub.resid, line="s", ax=axs[1], markersize=3)
axs[2].scatter(range(len(lm_sub.resid)), lm_sub.resid, s=8)
plt.show()
pg.normality(lm_sub.resid)


In [ ]:
data.groupby('Subject')[['Reaction']].mean()


In [ ]:
plt.scatter(data.Days, lm_sub.fittedvalues)


In [ ]:
#посмотрим на прогноз модели по каждому испытуемому
pred_ols = lm_sub.get_prediction()
data['fitted_sub'] = lm_sub.fittedvalues

fig, ax = plt.subplots(figsize=(15, 8))
ax.scatter(data.Days, data.Reaction, color="C0", ec="black", alpha=0.7)
sns.lineplot(x = 'Days', y = 'fitted_sub', hue = 'Subject', data=data)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))


### Mixed models

A mixed model is a statistical model that includes both fixed and random effects.

Fixed effects describe coefficients shared by the whole set of observations: for example, an averaged effect of a factor (drug dosage) or the level of a group (sex, experimental condition). Random effects describe deviations that vary from group to group (for example, the individual shift of each subject's reaction time) and are assumed to be drawn from some distribution. In a mixed model, fixed and random effects are estimated jointly. The division into fixed and random effects is fairly conventional and depends on the specific experiment: the same factor can enter the model differently depending on the research question.

#### Random-intercept model

The idea: every subject has their own baseline level of reaction time, while how much these shifts may vary is limited by the variance of the random effect. The model is formulated as

$$Reaction_{ij} = \beta_0 + \beta_1 Days_{ij} + b_i + \varepsilon_{ij},$$

where $i$ indexes subjects, $j$ indexes days, and

$$b_i \sim N(0, \sigma_b^2), \qquad \varepsilon_{ij} \sim N(0, \sigma^2).$$

In R the model is written as
```R
mod1 <- lmer(Reaction ~ Days + (1 | Subject), data = data)
```


In [ ]:
lm_mixed = smf.mixedlm("Reaction ~ Days", data, groups=data["Subject"]).fit()
print(lm_mixed.summary())


In [ ]:
lm_mixed.conf_int()


In [ ]:
# Стандартное отклонение случайного эффекта берём из подогнанной модели, а не из summary вручную
var_subject = lm_mixed.cov_re.iloc[0, 0]
print(f"Стандартное отклонение для распределения коэффициентов по субъектам: sqrt({var_subject:.1f}) = {np.sqrt(var_subject):.2f} мс")
print(f"Остаточное стандартное отклонение: sqrt({lm_mixed.scale:.1f}) = {np.sqrt(lm_mixed.scale):.2f} мс")
print(f"Внутриклассовая корреляция ICC = {var_subject / (var_subject + lm_mixed.scale):.2f}")


In [ ]:
print("Можно посмотреть, как по субъектам сдвигается время реакции")
pd.DataFrame(lm_mixed.random_effects).T.plot(kind='bar');


In [ ]:
#посмотрим на прогноз модели по каждому испытуемому
data['fitted_sub'] = lm_mixed.fittedvalues

fig, ax = plt.subplots(figsize=(15, 8))
ax.scatter(data.Days, data.Reaction, color="C0", ec="black", alpha=0.7)
sns.lineplot(x = 'Days', y = 'fitted_sub', hue = 'Subject', data=data)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))


In [ ]:
fig, axs = plt.subplots(1, 3, figsize = (10, 3))
axs[0].hist(lm_mixed.resid)
qqplot(lm_mixed.resid, line="s", ax=axs[1], markersize=3)
axs[2].scatter(range(len(lm_mixed.resid)), lm_mixed.resid, s=8)
plt.show()
pg.normality(lm_mixed.resid)


In [ ]:
print("Посмотрим, как устроены остатки модели по дням и субъектам: по дням остаётся систематический тренд")
data['resids'] = lm_mixed.resid
fig, axs = plt.subplots(1, 2, figsize = (10, 3))
sns.boxplot(x = 'Days', y = 'resids', data=data, ax=axs[0])
sns.boxplot(x = 'Subject', y = 'resids', data=data, ax=axs[1])
axs[1].tick_params(axis="x", labelrotation=90)
plt.show()


#### A confidence band for the population line

It is useful to construct a confidence band for the mean response $\mu(x) = \beta_0 + \beta_1 x$. An important subtlety: one must not plug in both lower (or both upper) confidence limits of $\beta_0$ and $\beta_1$ at once — that traces the corners of a parameter rectangle, ignores the covariance of the estimates and overstates uncertainty. The correct pointwise band uses the covariance matrix of the fixed effects:

$$\widehat{\mathrm{se}}\big(\hat\mu(x)\big) = \sqrt{\mathbf{x}^{\top} \, \widehat{\operatorname{Cov}}(\hat\beta)\, \mathbf{x}}, \qquad \hat\mu(x) \pm z_{0.975}\,\widehat{\mathrm{se}}.$$

Below, the incorrect "corner" band is shown in gray dotted lines for comparison.


In [ ]:
def fixed_effect_ci_band(res, xs):
    # Поточечный доверительный коридор для фиксированных эффектов MixedLM
    X = np.column_stack([np.ones_like(xs), xs])
    beta = res.fe_params.values[:2]
    V = res.cov_params().loc[["Intercept", "Days"], ["Intercept", "Days"]].values
    mean = X @ beta
    se = np.sqrt(np.einsum("ij,jk,ik->i", X, V, X))
    crit = stats.norm.ppf(0.975)
    return mean, mean - crit * se, mean + crit * se

xs = np.linspace(0, 9, 100)
mean, lo, hi = fixed_effect_ci_band(lm_mixed, xs)

# Некорректный «угловой» коридор для сравнения
conf_ints = lm_mixed.conf_int()
corner_lo = conf_ints.loc['Intercept', 0] + conf_ints.loc['Days', 0] * xs
corner_hi = conf_ints.loc['Intercept', 1] + conf_ints.loc['Days', 1] * xs

fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(data.Days, data.Reaction, color="C0", ec="black", alpha=0.35, s=18, label="data")
ax.plot(xs, mean, "r-", label="FixedMean")
ax.fill_between(xs, lo, hi, color="C3", alpha=0.3, label="корректный 95% коридор")
ax.plot(xs, corner_lo, ":", color="gray", lw=2, label="«угловой» коридор (некорректно)")
ax.plot(xs, corner_hi, ":", color="gray", lw=2)
ax.legend(loc="best")
plt.show()


#### Random intercept and random slope

The model has improved, but the residuals still show a problem: for some subjects the residuals trend systematically with days — each person has their own rate of slowing. Let us add a random slope:

$$Reaction_{ij} = \beta_0 + \beta_1 Days_{ij} + b_i + c_{i} Days_{ij} + \varepsilon_{ij},$$

where

$$b_i \sim N(0, \sigma_b^2), \qquad c_i \sim N(0, \sigma_c^2), \qquad \varepsilon_{ij} \sim N(0, \sigma^2).$$

In R the model is written as
```R
mod2 = lmer(Reaction ~ Days + (1 + Days | Subject), data = data)
```

In statsmodels the syntax is different: random slopes are added via `re_formula`. All the usual formula conventions still apply.


In [ ]:
lm_mixed = smf.mixedlm("Reaction ~ Days", data, groups=data["Subject"], re_formula="~ Days").fit()
print(lm_mixed.summary())


In [ ]:
print("Теперь у каждого испытуемого два параметра: собственный сдвиг и собственная чувствительность к дням недосыпа")

re = pd.DataFrame(lm_mixed.random_effects)
display(re)
re.T.plot(kind='bar');


With several random effects, the question of their association arises: if the intercept shift and the slope correlate, the covariance between them has to be modeled. The standard mixed model gives the pair $(b_i, c_i)$ a bivariate normal distribution with correlation $\rho$.

A common misconception is worth clarifying: statsmodels with `re_formula="~ Days"` **does estimate** the full random-effects covariance matrix — in the summary it appears as `Group x Days Cov`. That is, statsmodels estimates the correlation $\rho$ just as `lme4` does with the formula `(1 + Days | Subject)`. The limitation lies elsewhere: statsmodels provides no standard error or confidence interval for $\rho$ or for the other variance components.


In [ ]:
# Корреляция случайных эффектов из модели (а не из диаграммы рассеяния BLUP)
cov_re = lm_mixed.cov_re
print("Оценённая ковариационная матрица случайных эффектов:")
print(cov_re.round(2))

corr_model = cov_re.iloc[0, 1] / np.sqrt(cov_re.iloc[0, 0] * cov_re.iloc[1, 1])
print(f"\nМодельная оценка корреляции rho = {corr_model:.3f} (без стандартной ошибки)")
print(f"Выборочная корреляция BLUP-оценок (для сравнения) = {re.T['Group'].corr(re.T['Days']):.3f}")
print("BLUP — это условные моды, а не оценки параметров; их выборочная корреляция смешивает сжатие с истинной rho.")


In [ ]:
display(pg.corr(re.T['Group'], re.T['Days']))
sns.scatterplot(x='Group', y = 'Days', data=re.T)
plt.show()


In [ ]:
print("Хоть и видно отклонение от нормальности, но только за счёт нескольких выбросов, а в остальном остатки сильно лучше")
fig, axs = plt.subplots(1, 3, figsize = (10, 3))
axs[0].hist(lm_mixed.resid)
qqplot(lm_mixed.resid, line="s", ax=axs[1], markersize=3)
axs[2].scatter(range(len(lm_mixed.resid)), lm_mixed.resid, s=8)
plt.show()
pg.normality(lm_mixed.resid)


In [ ]:
print("Посмотрим, как устроены остатки модели по дням и субъектам: теперь всё сильно лучше")
data['resids'] = lm_mixed.resid
fig, axs = plt.subplots(1, 2, figsize = (10, 3))
sns.boxplot(x = 'Days', y = 'resids', data=data, ax=axs[0])
sns.boxplot(x = 'Subject', y = 'resids', data=data, ax=axs[1])
axs[1].tick_params(axis="x", labelrotation=90)
plt.show()


In [ ]:
print("Доверительный коридор изменился, в том числе по дням — это важно для выводов")
mean, lo, hi = fixed_effect_ci_band(lm_mixed, xs)
corner_lo = conf_ints.loc['Intercept', 0] + conf_ints.loc['Days', 0] * xs
corner_hi = conf_ints.loc['Intercept', 1] + conf_ints.loc['Days', 1] * xs

fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(data.Days, data.Reaction, color="C0", ec="black", alpha=0.35, s=18, label="data")
ax.plot(xs, mean, "r-", label="FixedMean")
ax.fill_between(xs, lo, hi, color="C3", alpha=0.3, label="корректный 95% коридор")
ax.plot(xs, corner_lo, ":", color="gray", lw=2, label="«угловой» коридор (некорректно)")
ax.plot(xs, corner_hi, ":", color="gray", lw=2)
ax.legend(loc="best")
plt.show()


## How to test hypotheses and choose models?

- For model choice, these are likelihood-based criteria (the likelihood itself, AIC, BIC).
- For the presence of an effect, one wants a confidence interval and a t-statistic — and here there are problems: they can be obtained for mixed models, but the approximations are rough. To avoid misleading users, the `lme4` package does not report significance levels at all.

What can be done? There is the parametric bootstrap.

One can also use Bayesian models: there the task is posed differently (Monte Carlo or variational inference), and the output consists of posterior distributions over the parameters.


## Bayesian hierarchical model

The theory is outside the scope of the course; if you are curious, see Richard McElreath (https://xcelab.net/rm/statistical-rethinking/): he has lectures, a book, videos. Here we only look at how it is done in practice.

Thanks to the https://bambinos.github.io/bambi/ package, building such models has become very easy. Formulas work the same way, and the syntax closely resembles `lme4`.

The main idea: all parameters — fixed effects and the variances of random effects alike — receive prior distributions, and sampling yields a posterior distribution for each of them.

One caveat worth knowing: in Bambi, group effects in a formula like `(Days | Subject)` are by default **independent** — their correlation is not estimated (unlike `lme4` and unlike the statsmodels mixed model above). For these data this is not critical: we have already seen that the model estimate of the correlation is small (ρ ≈ 0.07). If the correlation itself is the object of study, the correlated model is written by hand in PyMC.


In [ ]:
model1 = bmb.Model("Reaction ~ 1 + Days + (1 | Subject)", data, categorical="Subject")
model1


In [ ]:
idata1 = model1.fit(draws=1000, tune=1000, chains=4, random_seed=RANDOM_SEED,
                    idata_kwargs={"log_likelihood": True})


In [ ]:
az.summary(idata1)


In [ ]:
# Прежняя версия тетради печатала здесь WAIC; в актуальном ArviZ остался критерий PSIS-LOO
print(az.loo(idata1))


In [ ]:
model2 = bmb.Model("Reaction ~ 1 + Days + (Days | Subject)", data, categorical="Subject")
model2


In [ ]:
idata2 = model2.fit(draws=1000, tune=1000, chains=4, random_seed=RANDOM_SEED,
                    idata_kwargs={"log_likelihood": True})


In [ ]:
print(az.loo(idata2))


In [ ]:
models_dict = {
    "model1": idata1,
    "model2": idata2
}
df_compare = az.compare(models_dict)
df_compare


In [ ]:
az.plot_compare(df_compare)
plt.show()


In [ ]:
#проверка коррелированности параметров
_, ax = plt.subplots()

post_ds = idata2.posterior.to_dataset()
post_ds.plot.scatter(
    x="1|Subject", y="Days|Subject",
    hue="Subject__factor_dim",
    add_colorbar=False,
    add_legend=False,
    cmap="tab20",
    s=4, alpha=0.15,
)
ax.set_xlabel("апостериорный сдвиг интерсепта (мс)")
ax.set_ylabel("апостериорный сдвиг наклона (мс/день)")
plt.show()


In [ ]:
# Апостериорные интерсепты и наклоны каждого испытуемого с 94% интервалами
subjects = list(data["Subject"].unique())
b_draws = post_ds["1|Subject"].values.reshape(-1, len(subjects))
c_draws = post_ds["Days|Subject"].values.reshape(-1, len(subjects))

def forest(ax, draws, xlabel):
    order = np.argsort(draws.mean(axis=0))
    for pos, idx in enumerate(order):
        lo, hi = az.hdi(draws[:, idx])
        ax.hlines(pos, lo, hi, color="C0", lw=2, alpha=0.8)
        ax.plot(draws[:, idx].mean(), pos, "o", color="C3", ms=4)
    ax.set_yticks(range(len(order)))
    ax.set_yticklabels([str(subjects[i]) for i in order], fontsize=7)
    ax.set_xlabel(xlabel)

fig, axs = plt.subplots(1, 2, figsize = (10, 6))
forest(axs[0], b_draws, "сдвиг интерсепта (мс)")
axs[0].set_title("Интерсепт каждого испытуемого")
forest(axs[1], c_draws, "сдвиг наклона (мс/день)")
axs[1].set_title("Наклон по дням каждого испытуемого")
plt.tight_layout()
plt.show()


Note the difference from the classical model: each subject now has not a point estimate (a BLUP) but a full posterior distribution — with its own uncertainty. The same holds for the standard deviations of the random effects: the classical method gives one number without an interval, the Bayesian one gives a distribution.

# Homework
Build a mixed model for the pig growth data. These data and results can be found in most packages: https://www.rdocumentation.org/packages/doBy/versions/4.6-2/topics/data-dietox. Tutorials only consider the dependence on time, but you should also look at the other variables and try different models.

Data description:

This data frame contains the following columns:

- Weight: Weight
- Feed: Cumulated feed intake
- Time: Time (in weeks) in the experiment
- Pig: Id of each pig
- Evit: Vitamin E dose
- Cu: Copper dose
- Start: Start weight in experiment, i.e. weight at week 1
- Litter: Id of litter of each pig

Details: Data contains weight of slaughter pigs measured weekly for 12 weeks. Data also contains the start weight (i.e. the weight at week 1). The treatments are 3 different levels of Evit = vitamin E (dose: 0, 100, 200 mg dl-alpha-tocopheryl acetat /kg feed) in combination with 3 different levels of Cu = copper (dose: 0, 35, 175 mg/kg feed) in the feed. The cumulated feed intake is also recorded. The pigs are littermates.


In [ ]:
# Данные dietox — каноническая копия в репозитории курса
PIGS_PATH = "data/dietox.csv"
if not os.path.exists(PIGS_PATH):
    url_p = "https://raw.githubusercontent.com/iknyazeva/bayes-cogsci-book/main/data/dietox.csv"
    urllib.request.urlretrieve(url_p, PIGS_PATH)
data_pigs = pd.read_csv(PIGS_PATH)


In [ ]:
data_pigs.head()
